# Triage_HF Statistical & Graphical Analytical Model
* Started date: 02/07/2024 - 05:09 AM
* Data Management Team, Triage_HF

# 1. Dataset cleaning
We will import our dataset and perform a first cleanup to begin to understand which columns and values we are dealing with.

In [115]:
import numpy as np
import pandas as pd
import plotly.express as px
import datetime as dt

pd.options.mode.chained_assignment = None
pd.set_option('display.max.rows', None)
pd.set_option('display.max.columns', None)

In [116]:
train_df = pd.read_csv('../dataset/raw/TRIAGE 2024.csv')

In [117]:
train_df.head()

,3+-99999|a,[ñ_MJ,NOMBRE Y APELLIDO,MOTIVO DE CONSULTA,BOX,TRIAGE,ENFERMERO,MEDICO,DESTINO,A,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24
0,FECHA: 01/01/2024 ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,01-01,LUSI,FIEBRE Y TOS,18,IV,ERIKA,RODRIGO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,01-01,LO PINTO CARLOS,FIEBRE Y TOS,17,IV,SOLEDAD G,SOLEDAD,alta,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,01-01,BANDERA,TOS,12,IV,ERIKA,RODRIGO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,01-01,TOMINO EDUARDO,HTA,12,IV,SOLEDAD G,SOLEDAD,ALTA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


At first, you can see how certain columns are wrongly named, so I will assign a corresponding name to them:

In [118]:
cols_rename = { '[ñ_MJ': 'FECHA DE INGRESO',
                'A': 'AISLADO' }
train_df.rename(columns = cols_rename, inplace = True)

In addition, each time you move to the next day according to the date of entry, the first column returns to 1:

In [119]:
train_df.loc[train_df['FECHA DE INGRESO'] == '01-01', '3+-99999|a'].iloc[0] == train_df.loc[train_df['FECHA DE INGRESO'] == '01-02', '3+-99999|a'].iloc[0]

True

Therefore, I will name that first column "NUMERO DE TURNO" in reference to the fact that the turns are reset at the beginning of the next day:

In [120]:
col_rename = { '3+-99999|a': 'NUMERO DE TURNO' }
train_df.rename(columns = col_rename, inplace = True)

We will remove the header that appears every time a new day begins:

In [121]:
train_df = train_df[train_df['NUMERO DE TURNO'].str.contains('FECHA') == False]

For possible machine learning models in the future, the triage level should be int dtype:

In [122]:
train_df['TRIAGE'].dtype

dtype('O')

In [123]:
train_df['TRIAGE'] = train_df['TRIAGE'].str.strip().str.upper()

In [124]:
vals_rename = { 'I': 1, 'II': 2, 'III': 3, 'IV': 4 }
train_df['TRIAGE'] = train_df['TRIAGE'].replace(vals_rename)

In [125]:
train_df['TRIAGE'] = pd.to_numeric(train_df['TRIAGE'], downcast = "signed", errors = 'coerce').fillna(0)

In [126]:
train_df['TRIAGE'].dtype

dtype('float64')

We will change the object type of the entry date to date type. This makes date manipulation much easier and more readable:

In [127]:
train_df['FECHA DE INGRESO'].dtype

dtype('O')

In [128]:
train_df['FECHA DE INGRESO'] = train_df['FECHA DE INGRESO'] + '-2024'

In [129]:
train_df['FECHA DE INGRESO'] = pd.to_datetime(train_df['FECHA DE INGRESO'], format = '%d-%m-%Y', errors = 'coerce')

In [130]:
train_df['FECHA DE INGRESO'].dtype

dtype('<M8[ns]')

Finally, empty and unnamed columns can be visualized. We will remove them:

In [131]:
cols_to_keep = ['NUMERO DE TURNO', 'FECHA DE INGRESO', 'NOMBRE Y APELLIDO', 'MOTIVO DE CONSULTA', 'BOX', 'TRIAGE', 'ENFERMERO', 'MEDICO', 'DESTINO', 'AISLADO']
train_df = train_df[cols_to_keep]

This is how our dataframe would look at first:

In [132]:
train_df.head()

,NUMERO DE TURNO,FECHA DE INGRESO,NOMBRE Y APELLIDO,MOTIVO DE CONSULTA,BOX,TRIAGE,ENFERMERO,MEDICO,DESTINO,AISLADO
1,1,2024-01-01,LUSI,FIEBRE Y TOS,18,4.0,ERIKA,RODRIGO,NaN,NaN
2,2,2024-01-01,LO PINTO CARLOS,FIEBRE Y TOS,17,4.0,SOLEDAD G,SOLEDAD,alta,NaN
3,3,2024-01-01,BANDERA,TOS,12,4.0,ERIKA,RODRIGO,NaN,NaN
4,4,2024-01-01,TOMINO EDUARDO,HTA,12,4.0,SOLEDAD G,SOLEDAD,ALTA,NaN
5,5,2024-01-01,D IORIO ROLANDO EMILIO,FIEBRE,5,4.0,ERIKA,SOLEDAD,NaN,NaN


In [133]:
def turnos_fecha(desde, hasta):
    """
    This function receives a date range, and shows how many patients for each day of that date range were entered into the system.

    Args:
        desde (object): Beginning of the interval, is a date in d-m-yyyy format.
        hasta (object): End of the interval, is a date in d-m-yyyy format.
    """
    desde = pd.to_datetime(desde, format='%d-%m-%Y', errors='coerce') 
    hasta = pd.to_datetime(hasta, format='%d-%m-%Y', errors='coerce')
    train_df_acotado = train_df[(train_df['FECHA DE INGRESO'] >= desde) & (train_df['FECHA DE INGRESO'] <= hasta)]
    train_df_acotado['FECHA DE INGRESO'] = train_df_acotado['FECHA DE INGRESO'].dt.strftime('%d-%m-%Y')
    train_df_acotado = train_df_acotado.groupby('FECHA DE INGRESO', sort=False).size().reset_index()
    train_df_acotado.rename(columns={0: 'CANTIDAD DE PACIENTES'}, inplace=True)
    fig = px.bar(train_df_acotado, x='FECHA DE INGRESO', y='CANTIDAD DE PACIENTES', barmode="group")
    fig.show()

In [134]:
turnos_fecha('9-01-2024', '06-2-2024')